In [0]:
from pyspark.sql.functions import col,trim,lower,upper,when,current_timestamp,row_number
from pyspark.sql.window import Window

bronze_df = spark.read.table("`reconciliation-platform`.bronze.orders")

cleaned_df = (bronze_df
    .withColumn("order_id",trim(col("order_id")))
    .withColumn("customer_id",trim(col("customer_id")))
    .withColumn("order_date",col("order_date").cast("timestamp"))
    .withColumn("order_status",lower(trim(col("order_status"))))
    .withColumn("payment_method",lower(trim(col("payment_method"))))
    .withColumn("gross_amount",col("gross_amount").cast("double"))
    .withColumn("discount_amount",col("discount_amount").cast("double"))
    .withColumn("shipping_fee",col("shipping_fee").cast("double"))
    .withColumn("tax_amount",col("tax_amount").cast("double"))
    .withColumn("net_amount",col("net_amount").cast("double"))
    .withColumn("currency",lower(trim(col("currency"))))
    .withColumn("updated_at",col("updated_at").cast("timestamp"))
    .withColumn(
        "order_status",
        when(col("order_status").isin("complete","completed"),"completed")
        .when(col("order_status").isin("cancel","cancelled","canceled"),"cancelled")
        .when(col("order_status").isin("pending","processing"),"processing")
        .when(col("order_status").isin("ship","shipped"),"shipped")
        .when(col("order_status").isin("deliver","delivered"),"delivered")
        .when(col("order_status").isin("fail","failed"),"failed")
        .otherwise("unknown")
    )
    .withColumn(
        "payment_method",
        when(col("payment_method").isin("upi"),"upi")
        .when(col("payment_method").isin("card","credit card","debit card"),"card")
        .when(col("payment_method").isin("net banking","netbanking"),"netbanking")
        .when(col("payment_method").isin("cash on delivery","cash_on_delivery"),"cod")
        .when(col("payment_method").isin("wallet"),"wallet")
        .otherwise("unknown")
    )
    .withColumn(
        "currency",
        when(col("currency").isin("inr","rs","rs.","rupee","rupees"),"INR")
        .otherwise(upper(col("currency")))
    )
    .withColumn("discount_amount",when(col("discount_amount").isNull(),0.0).otherwise(col("discount_amount")))
    .withColumn("shipping_fee",when(col("shipping_fee").isNull(),0.0).otherwise(col("shipping_fee")))
    .withColumn("tax_amount",when(col("tax_amount").isNull(),0.0).otherwise(col("tax_amount")))
    .filter(col("order_id").isNotNull())
    .filter(col("order_id") != "")
    .filter(col("customer_id").isNotNull())
    .filter(col("customer_id") != "")
    .filter(col("gross_amount").isNotNull())
    .filter(col("gross_amount") >= 0)
    .filter(col("discount_amount") >= 0)
    .filter(col("shipping_fee") >= 0)
    .filter(col("tax_amount") >= 0)
    .withColumn(
        "net_amount",
        when(
            col("net_amount").isNull(),
            col("gross_amount") - col("discount_amount") + col("shipping_fee") + col("tax_amount")
        ).otherwise(col("net_amount"))
    )
)

window_spec = Window.partitionBy("order_id").orderBy(
    col("updated_at").desc_nulls_last(),
    col("_ingestion_timestamp").desc()
)

silver_df = (cleaned_df
    .withColumn("_row_number",row_number().over(window_spec))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .withColumn("_silver_processed_at",current_timestamp())
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`reconciliation-platform`.silver.orders"))